In [1]:
import time
import pandas as pd

import os
from pathlib import Path
import sys

import json
sys.path.insert(0, r'../../../quetzal')

from quetzal.os.parallel_call import parallel_call_notebook, parallel_call_notebooks
from quetzal.io import excel

qpath = Path(r'../../')

scenario_list = []

parameter_frame = pd.read_excel(
    qpath / r'inputs/parameters.xlsx'
).set_index(['category','parameter']).dropna(how='all')

scenario_list = [c for c in parameter_frame.columns if c not in ['description', 'unit']]

#workers = min(3, len(scenario_list)*6)
workers = 4

print('Scénarios disponibles : ', scenario_list)
print(workers)

Scénarios disponibles :  ['base', 'base_timetables']
4


## Choix des scénarios à lancer <img src="images\scenarios.png" style="width: 50px;" align="right"/>

Renseigner dans la cellule ci-dessous le nom des scénarios à lancer, entre guillements et espacés de virgules.

In [2]:
scenario_list = ["base"]

## Lancement du modèle <img src="images\runs.png" style="width: 50px;" align="right"/>

### Création des dossiers correspondants

In [3]:
workers = 8

In [4]:
scenario_list = [s for s in scenario_list]

print('Préparation du lancement des scénarios :', scenario_list)
print('Nombre de sous-process utilisés :', workers)

kwargs = {
    'workers':workers, 'errout_suffix':True, 'stdout_path':'../_logs/out.txt',
    'stderr_path':'../_logs/err.txt', 'sleep':1
}
os.makedirs('../_logs', exist_ok=True)

scen_args = []
base_args = []
for s in scenario_list:
    if "base" in s:
        base_args += [json.dumps({'scenario': s})]
    else:
        scen_args += [json.dumps({'scenario': s})]

    (qpath / r'scenarios/{}/model/'.format(s)).mkdir(parents=True, exist_ok=True)
    (qpath / r'scenarios/{}/outputs/'.format(s)).mkdir(parents=True, exist_ok=True)

args = scen_args + base_args
    

Préparation du lancement des scénarios : ['base']
Nombre de sous-process utilisés : 8


### Preparation du GTFS:
- Nettoyage du GTFS
- calcul des fréquences
- ajout du metro

In [5]:
parallel_call_notebook('A00_networks.ipynb', args, **kwargs)

../_logs/err_base_A00_networks.txt
0 A00_networks.py {"scenario": "base"}
22 seconds


In [6]:
parallel_call_notebook('A01_zones.ipynb', args, **kwargs)

../_logs/err_base_A01_zones.txt
0 A01_zones.py {"scenario": "base"}
10 seconds


In [7]:
parallel_call_notebook('A10_pathfinders.ipynb', args, **kwargs)

../_logs/err_base_A10_pathfinders.txt
0 A10_pathfinders.py {"scenario": "base"}
141 seconds


In [8]:
parallel_call_notebook('A20_generation_distribution.ipynb', args, **kwargs)

../_logs/err_base_A20_generation_distribution.txt
0 A20_generation_distribution.py {"scenario": "base"}
20 seconds


In [9]:
parallel_call_notebook('A30_modal_split_assignment.ipynb',  args, **kwargs)

../_logs/err_base_A30_modal_split_assignment.txt
0 A30_modal_split_assignment.py {"scenario": "base"}
subprocess **A30_modal_split_assignment.py0 {"scenario": "base"}** terminated with an error.
160 seconds
